# LIBRAIRIES

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from affine import Affine
from scipy.ndimage import gaussian_filter
from rasterio.features import rasterize, geometry_mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches
import fiona
import rasterio
from rasterio.enums import MergeAlg
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import libpysal
import esda
import rasterstats
import warnings
warnings.filterwarnings("ignore")
import math
from statsmodels.nonparametric.smoothers_lowess import lowess

import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from utils.config import *
from utils.functions import *

# PATH

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

# ─── Paths ──────────────────────────────────────────────────────────────────
input_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/INPUT/"
output_file_path_PL = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/"
input_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/INPUT/"
output_file_path_PL_wave1 = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/WAVE1_MOBILITY/OUTPUT/"
output_file_path_PL_RASTER = "/Volumes/T7_lin_win/PANEL_LEMANIQUE/GPS_tracking_data_PL/OUTPUT/RASTER/"

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/GE/step-1/'
output_step2_path='../../Data/output/GE/step-2/'
output_step3_path='../../Data/output/GE/step-3/'

# IMPORTS

In [ ]:
# ─── Legs ─────────────────────────────────────────────────────────────────────
legs_GE_all          = gpd.read_parquet(f'{output_file_path_PL}legs_GE_all.parquet')
legs_GE_walk         = gpd.read_parquet(f'{output_file_path_PL}legs_GE_walk.parquet')
legs_GE_walk_regular = gpd.read_parquet(f'{output_file_path_PL}legs_GE_walk_regular.parquet')

# ─── Users ────────────────────────────────────────────────────────────────────
users_GE              = pd.read_csv(f'{output_file_path_PL}users_GE.csv')
users_GE_walk         = pd.read_csv(f'{output_file_path_PL}users_GE_walk.csv')
users_GE_walk_regular = pd.read_csv(f'{output_file_path_PL}users_GE_walk_regular.csv')

print("✓ données chargées")

In [ ]:
canton_GE  = gpd.read_file(f'{input_file_path}/network_agreg/CANTON_GE/CANTON_POLYGON.shp')
canton_GE = canton_GE.to_crs(operation_crs)

user_stat  = pd.read_csv(f'{input_file_path_PL}...')

girec     = gpd.read_file(input_file_path_PL + "GEO_GIREC-SHP/GEO_GIREC.shp").to_crs(epsg=2056)
communes  = gpd.read_file(input_file_path_PL + "CAD_COMMUNE-SHP/CAD_COMMUNES_GE_fusionnee.shp").to_crs(epsg=2056)
lac_leman = gpd.read_file(input_file_path_PL + "LAC_LEMAN_WITHOUT_BRIDGE-SHP/LAC_LEMAN_WITHOUT_BRIDGE.shp").to_crs(epsg=2056)

# MAP VISUALISATION

## LEGS DENSITY X WALKABILITY

### LEGS GAUSSIAN FILTER

Permet d'avoir la visualisation des traces de marchabilité et aussi l'information de densité de marche aux différents endrois de la carte (rasterisation via pixel de 10x10m)

### TEST V1 DENSITE

In [ ]:
# ─── Reproject everything to EPSG:2056 (meters) ───────────────────────────────
legs_GE_walk_regular = legs_GE_walk_regular.to_crs(epsg=2056)

if canton_GE.crs.to_epsg() != 2056:
    canton_GE = canton_GE.to_crs(epsg=2056)

# ─── Check ────────────────────────────────────────────────────────────────────
print(f"legs CRS   : {legs_GE_walk_regular.crs}")
print(f"canton CRS : {canton_GE.crs}")

print(f"GIREC : {len(girec)} features, CRS: {girec.crs}")

# ─── Define profile ───────────────────────────────────────────────────────────
profile    = ""  # ← "" pour tous, "senior", "young", "middle"
age_filter = profile_age_map.get(profile, "")

print(f"Profile : {profile}")

if age_filter is not None:
    gdf = legs_GE_walk_regular[
        (legs_GE_walk_regular.geometry.notna()) &
        (legs_GE_walk_regular["age_fr_grouped"].isin(age_filter))
    ].copy()
    print(f"Age filter    : {age_filter}")
else:
    gdf = legs_GE_walk_regular[legs_GE_walk_regular.geometry.notna()].copy()
    print(f"Age filter    : none (all users)")

print(f"Features to rasterize : {len(gdf)}")
print(f"Unique users          : {gdf['user_id_fors'].nunique()}")



#### TEST RASTER DENSITE PASSAGE PIETON

In [ ]:
print("Creating raster...")

pixel_size = 10

# ─── Grille basée sur le canton (sans le lac) ─────────────────────────────────
# → grille d'analyse ancrée sur le territoire officiel, indépendante des données
xmin, ymin, xmax, ymax = canton_GE.total_bounds  # ← canton au lieu de legs

raster_width  = int((xmax - xmin) / pixel_size)
raster_height = int((ymax - ymin) / pixel_size)

transform = Affine(pixel_size, 0, xmin,
                   0, -pixel_size, ymax)

print(f"raster_width={raster_width}, raster_height={raster_height}")

In [ ]:
print("Clipping to canton and remove Léman lake...")

canton_mask = geometry_mask(
    geometries=canton_GE.geometry,
    out_shape=(raster_height, raster_width),
    transform=transform,
    invert=True
)

lac_mask = geometry_mask(
    geometries=lac_leman.geometry,
    out_shape=(raster_height, raster_width),
    transform=transform,
    invert=True
)

# Masque combiné : dans le canton ET pas dans le lac
canton_GE_mask = canton_mask & ~lac_mask


In [ ]:
# ─── Visualisation du masque canton ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(canton_GE_mask, origin='upper', extent=[xmin, xmax, ymin, ymax],
          cmap='Greys', alpha=0.8)
canton_GE.boundary.plot(ax=ax, color='red', linewidth=1.5)
#ax.set_title(f"Canton mask — {canton_GE_mask.sum():,} pixels dans le canton")
ax.set_axis_off()
plt.tight_layout()
plt.show()

print(f"Canton mask — {canton_GE_mask.sum():,} pixels dans le canton")

In [ ]:
with rasterio.open(
    f"{output_file_path_PL_RASTER}/canton_GE_mask.tif",
    mode      = "w",
    driver    = "GTiff",
    height    = raster_height,
    width     = raster_width,
    count     = 1,
    dtype     = "uint8",        # booléen → 0 ou 1
    crs       = "EPSG:2056",
    transform = transform,
    nodata    = 255
) as dst:
    dst.write(canton_GE_mask.astype("uint8"), 1)

print(f"canton_GE_mask exporté")

In [ ]:
extent = [xmin, xmax, ymin, ymax]

#### RASTER DENSITE PASSAGE PIETON PAR TIMESLOT

### TEST V2 DENSITE

In [ ]:
# ─── User test ────────────────────────────────────────────────────────────────
test_user_id = "CH8943"  # ← remplace par l'identifiant du user

In [ ]:
# ─── Isoler les traces du user ────────────────────────────────────────────────
gdf_user = gdf[gdf["user_id_fors"] == test_user_id].copy()

n_days_GE_user = gdf_user["n_days_GE"].iloc[0]

print(f"User        : {test_user_id}")
print(f"n_legs      : {len(gdf_user)}")
print(f"n_days_GE   : {n_days_GE_user}")

# ─── Rasterisation brute ──────────────────────────────────────────────────────
raster_user_raw = rasterize(
    shapes=((geom, 1) for geom in gdf_user.geometry if geom is not None),
    out_shape=(raster_height, raster_width),
    transform=transform,
    fill=0,
    dtype='float32',
    merge_alg=MergeAlg.add
)

print(f"\nraster_user_raw shape  : {raster_user_raw.shape}")
print(f"raster_user_raw min    : {raster_user_raw.min():.1f}")
print(f"raster_user_raw max    : {raster_user_raw.max():.1f}")
print(f"pixels > 0             : {(raster_user_raw > 0).sum()}")

# ─── Clipping canton ──────────────────────────────────────────────────────────
raster_user_raw_clipped = raster_user_raw.copy().astype('float32')
raster_user_raw_clipped[~canton_GE_mask] = np.nan

# ─── Zoom automatique sur les traces du user ──────────────────────────────────
user_bounds = gdf_user.total_bounds
zoom_margin = 500

xmin_u, ymin_u, xmax_u, ymax_u = user_bounds

# ─── Normalisation pour la visu ───────────────────────────────────────────────
d_max_user = compute_scale([raster_user_raw_clipped], clip_percentile, mode=norm_mode)
raster_user_raw_norm = normalize_raster(raster_user_raw_clipped, d_max_user, mode=norm_mode)

plot_density_map(
    raster_user_raw_norm,
    title=f"",   #f"User test — traces brutes (zoom)\nn_legs={len(gdf_user)} | n_days_GE={n_days_GE_user}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_u, ymin_u, xmax_u, ymax_u),
    margin=zoom_margin
)

In [ ]:
# ─── Plot brut pour le rapport ────────────────────────────────────────────────
r_plot = raster_user_raw_clipped.copy()
r_plot[r_plot == 0] = np.nan  # 0 → NaN pour afficher en blanc

fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

im = ax.imshow(
    r_plot,
    origin='upper',
    extent=extent,
    cmap=cmap_plot,
    alpha=0.8,
    vmin=0,
    vmax=np.nanmax(r_plot)   # échelle explicite en passages bruts
)

canton_GE.boundary.plot(ax=ax, color='black', linewidth=1.5)
girec.boundary.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.4)

ax.set_xlim(xmin_u - zoom_margin, xmax_u + zoom_margin)
ax.set_ylim(ymin_u - zoom_margin, ymax_u + zoom_margin)
ax.set_axis_off()

cbar = plt.colorbar(
    im, ax=ax,
    fraction=0.03, pad=0.01,
)
cbar.set_label(r"Raw passage count $C_u(x,y)$ [legs]", fontsize=14)
cbar.ax.tick_params(labelsize=14)

plt.tight_layout()
#plt.savefig("6.Methodology/raster_step1_example.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Normalisation par n_days_GE ──────────────────────────────────────────────
raster_user_daily = raster_user_raw / n_days_GE_user

print(f"n_days_GE              : {n_days_GE_user}")
print(f"raster_user_daily min  : {raster_user_daily.min():.4f}")
print(f"raster_user_daily max  : {raster_user_daily.max():.4f}")
print(f"pixels > 0             : {(raster_user_daily > 0).sum()}")

# ─── Clipping canton ──────────────────────────────────────────────────────────
raster_user_daily_clipped = raster_user_daily.copy()
raster_user_daily_clipped[~canton_GE_mask] = np.nan

# ─── Plot ─────────────────────────────────────────────────────────────────────
d_max_daily = compute_scale([raster_user_daily_clipped], clip_percentile, mode=norm_mode)
raster_user_daily_norm = normalize_raster(raster_user_daily_clipped, d_max_daily, mode=norm_mode)

plot_density_map(
    raster_user_daily_norm,
    title=f"User test — fréquentation journalière\nn_legs={len(gdf_user)} | n_days_GE={n_days_GE_user}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_u, ymin_u, xmax_u, ymax_u),
    margin=zoom_margin
)

In [ ]:
# ─── Plot brut journalier pour le rapport ─────────────────────────────────────
r_daily_plot = raster_user_daily_clipped.copy()
r_daily_plot[r_daily_plot == 0] = np.nan

fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

im = ax.imshow(
    r_daily_plot,
    origin='upper',
    extent=extent,
    cmap=cmap_plot,
    alpha=0.8,
    vmin=0,
    vmax=np.nanmax(r_daily_plot)
)

canton_GE.boundary.plot(ax=ax, color='black', linewidth=1.5)
girec.boundary.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.4)

ax.set_xlim(xmin_u - zoom_margin, xmax_u + zoom_margin)
ax.set_ylim(ymin_u - zoom_margin, ymax_u + zoom_margin)
ax.set_axis_off()

cbar = plt.colorbar(
    im, ax=ax,
    fraction=0.03, pad=0.01,
)
cbar.set_label(r"Daily passage rate $C_u^{\mathrm{daily}}(x,y)$ [legs$\cdot$day$^{-1}$]", fontsize=14)
cbar.ax.tick_params(labelsize=14)

plt.tight_layout()
#plt.savefig("6.Methodology/raster_step2_example.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Lissage gaussien ─────────────────────────────────────────────────────────
sigma = 1  # ← ajustable si besoin

raster_user_smoothed = gaussian_filter(raster_user_daily, sigma=sigma)

print(f"sigma                    : {sigma}")
print(f"raster_user_smoothed min : {raster_user_smoothed.min():.4f}")
print(f"raster_user_smoothed max : {raster_user_smoothed.max():.4f}")
print(f"pixels > 0 avant lissage : {(raster_user_daily > 0).sum()}")
print(f"pixels > 0 après lissage : {(raster_user_smoothed > 0).sum()}")

# ─── Clipping canton ──────────────────────────────────────────────────────────
raster_user_smoothed_clipped = raster_user_smoothed.copy()
raster_user_smoothed_clipped[~canton_GE_mask] = np.nan

# ─── Plot ─────────────────────────────────────────────────────────────────────
d_max_smoothed = compute_scale([raster_user_smoothed_clipped], clip_percentile, mode=norm_mode)
raster_user_smoothed_norm = normalize_raster(raster_user_smoothed_clipped, d_max_smoothed, mode=norm_mode)

plot_density_map(
    raster_user_smoothed_norm,
    title=f"User test — fréquentation journalière lissée (sigma={sigma})\nn_legs={len(gdf_user)} | n_days_GE={n_days_GE_user}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_u, ymin_u, xmax_u, ymax_u),
    margin=zoom_margin
)

In [ ]:
r_smoothed_plot = raster_user_smoothed_clipped.copy()
r_smoothed_plot[r_smoothed_plot == 0] = np.nan

fig, ax = plt.subplots(figsize=(10, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

im = ax.imshow(
    r_smoothed_plot,
    origin='upper',
    extent=extent,
    cmap=cmap_plot,
    alpha=0.8,
    vmin=0,
    vmax=np.nanmax(r_smoothed_plot)
)

canton_GE.boundary.plot(ax=ax, color='black', linewidth=1.5)
girec.boundary.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.4)

ax.set_xlim(xmin_u - zoom_margin, xmax_u + zoom_margin)
ax.set_ylim(ymin_u - zoom_margin, ymax_u + zoom_margin)
ax.set_axis_off()


cbar = plt.colorbar(
    im, ax=ax,
    fraction=0.03, pad=0.01,
)
cbar.set_label(r"Smoothed daily passage rate $R_u(x,y)$ [legs$\cdot$day$^{-1}$]", fontsize=14)
cbar.ax.tick_params(labelsize=14)


plt.tight_layout()
#plt.savefig("6.Methodology/raster_step3_example.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── User test ────────────────────────────────────────────────────────────────
test_user_id = "CH8943"  # ← remplace par l'identifiant du user

In [ ]:
(legs_GE_walk_regular["user_id_fors"] == "CH8943").sum()

In [ ]:
# ─── Calcul n_days_per_user ───────────────────────────────────────────────────
n_days_per_user = (
    legs_GE_walk_regular
    .groupby("user_id_fors")["n_days_GE"]
    .first()
    .to_dict()
)

print(f"n_days_per_user : {len(n_days_per_user)} users")
print(f"n_days médiane  : {np.median(list(n_days_per_user.values())):.0f}")
print(f"n_days max      : {max(n_days_per_user.values())}")
print(f"n_days min      : {min(n_days_per_user.values())}")

In [ ]:
gdf_one_user = gdf[gdf["user_id_fors"] == test_user_id].copy()

raster_test, meta_test = build_density_raster_v2(
    gdf_group       = gdf_one_user,
    raster_height   = raster_height,
    raster_width    = raster_width,
    transform       = transform,
    canton_mask     = canton_GE_mask,
    n_days_per_user = n_days_per_user,  # le dict complet, la fonction ira chercher ce user
    sigma           = 0,
    verbose         = True,
    output_path     = output_file_path_PL_RASTER,  # pas d'export si tu veux juste tester
    raster_name     = 'test_user_density',
)

print(meta_test)

In [ ]:
test_users = ["CH1118", "CH8948"]

In [ ]:
gdf_two_users = gdf[gdf["user_id_fors"].isin(test_users)].copy()

raster_test_2, meta_test_2 = build_density_raster_v2(
    gdf_group       = gdf_two_users,
    raster_height   = raster_height,
    raster_width    = raster_width,
    transform       = transform,
    canton_mask     = canton_GE_mask,
    n_days_per_user = n_days_per_user,
    sigma           = 0,
    verbose         = True,
    output_path     = output_file_path_PL_RASTER,
    raster_name     = 'test_2_users'
)

In [ ]:
# ─── Isoler les traces des users test ────────────────────────────────────────
test_user_ids = ["CH15203", "CH14265"] #liste d'id à tester

In [ ]:
# ─── Calcul n_days_per_user ───────────────────────────────────────────────────
n_days_per_user = (
    legs_GE_walk_regular
    .groupby("user_id_fors")["n_days_GE"]
    .first()
    .to_dict()
)

print(f"n_days_per_user : {len(n_days_per_user)} users")
print(f"n_days médiane  : {np.median(list(n_days_per_user.values())):.0f}")
print(f"n_days max      : {max(n_days_per_user.values())}")
print(f"n_days min      : {min(n_days_per_user.values())}")

In [ ]:
gdf_user = gdf[gdf["user_id_fors"].isin(test_user_ids)].copy()

print(f"Users   : {test_user_ids}")
print(f"n_legs  : {len(gdf_user)}")
for uid in test_user_ids:
    n = n_days_per_user.get(uid, None)
    print(f"  {uid} → n_days_GE={n}")

# ─── Rasterisation brute ──────────────────────────────────────────────────────
raster_user_raw = rasterize(
    shapes=((geom, 1) for geom in gdf_user.geometry if geom is not None),
    out_shape=(raster_height, raster_width),
    transform=transform,
    fill=0,
    dtype='float32',
    merge_alg=MergeAlg.add
)

print(f"\nraster_user_raw shape : {raster_user_raw.shape}")
print(f"raster_user_raw min   : {raster_user_raw.min():.1f}")
print(f"raster_user_raw max   : {raster_user_raw.max():.1f}")
print(f"pixels > 0            : {(raster_user_raw > 0).sum()}")

# ─── Clipping canton ──────────────────────────────────────────────────────────
raster_user_raw_clipped = raster_user_raw.copy().astype('float32')
raster_user_raw_clipped[~canton_GE_mask] = np.nan

# ─── Zoom automatique ────────────────────────────────────────────────────────
user_bounds = gdf_user.total_bounds
zoom_margin = 500
xmin_u, ymin_u, xmax_u, ymax_u = user_bounds

# ─── Normalisation et plot ────────────────────────────────────────────────────
d_max_user       = compute_scale([raster_user_raw_clipped], clip_percentile, mode=norm_mode)
raster_user_raw_norm = normalize_raster(raster_user_raw_clipped, d_max_user, mode=norm_mode)

plot_density_map(
    raster_user_raw_norm,
    title=f"Users test — traces brutes (zoom)\nn_legs={len(gdf_user)}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_u, ymin_u, xmax_u, ymax_u),
    margin=zoom_margin
)

In [ ]:
# ─── Mini boucle test — moyenne sur les users test ───────────────────────────
sigma         = 1
raster_sum_test  = np.zeros((raster_height, raster_width), dtype='float32')
n_users_test  = 0

for uid in test_user_ids:
    gdf_u  = gdf[gdf["user_id_fors"] == uid]
    n_days = n_days_per_user.get(uid, None)

    if n_days is None or n_days == 0:
        print(f"  {uid} → skippé (n_days invalide)")
        continue

    r = rasterize(
        shapes=((geom, 1) for geom in gdf_u.geometry if geom is not None),
        out_shape=(raster_height, raster_width),
        transform=transform,
        fill=0,
        dtype='float32',
        merge_alg=MergeAlg.add
    )

    r_daily    = r / n_days
    r_smoothed = gaussian_filter(r_daily, sigma=sigma)

    print(f"  {uid} → n_days={n_days} | n_legs={len(gdf_u)} | max={r_smoothed.max():.4f}")

    raster_sum_test += r_smoothed
    n_users_test    += 1

# ─── Moyenne ──────────────────────────────────────────────────────────────────
raster_mean_test         = raster_sum_test / n_users_test
raster_mean_test_clipped = raster_mean_test.copy()
raster_mean_test_clipped[~canton_GE_mask] = np.nan

print(f"\nn_users_test : {n_users_test}")
print(f"max          : {np.nanmax(raster_mean_test_clipped):.6f}")
print(f"pixels > 0   : {(raster_mean_test_clipped > 0).sum()}")

# ─── Plot ─────────────────────────────────────────────────────────────────────
d_max_test       = compute_scale([raster_mean_test_clipped], clip_percentile, mode=norm_mode)
raster_mean_test_norm = normalize_raster(raster_mean_test_clipped, d_max_test, mode=norm_mode)

plot_density_map(
    raster_mean_test_norm,
    title=f"Users test — moyenne journalière lissée (sigma={sigma})\nn_users={n_users_test}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_u, ymin_u, xmax_u, ymax_u),
    margin=zoom_margin
)

In [ ]:
# ─── Calcul n_days_per_user ───────────────────────────────────────────────────
n_days_per_user = (
    legs_GE_walk_regular
    .groupby("user_id_fors")["n_days_GE"]
    .first()
    .to_dict()
)

print(f"n_days_per_user : {len(n_days_per_user)} users")
print(f"n_days médiane  : {np.median(list(n_days_per_user.values())):.0f}")
print(f"n_days max      : {max(n_days_per_user.values())}")
print(f"n_days min      : {min(n_days_per_user.values())}")

In [ ]:
n_days_per_user

In [ ]:
# ─── Load communes ────────────────────────────────────────────────────────────
focus_commune_name = "Genève"
focus_commune      = communes[communes["COMMUNE"] == focus_commune_name]
xmin_z, ymin_z, xmax_z, ymax_z = focus_commune.total_bounds
margin = 200

print(f"Focus commune : {focus_commune_name}")
print(f"Bounds        : {focus_commune.total_bounds}")

In [ ]:
# ─── Test build_density_raster_v2 sur tous les users ─────────────────────────
raster_mean_clipped_v2, meta_v2 = build_density_raster_v2(
    gdf_group       = gdf,
    raster_height   = raster_height,
    raster_width    = raster_width,
    transform       = transform,
    canton_mask     = canton_GE_mask,
    n_days_per_user = n_days_per_user,
    sigma           = 1,
    verbose         = True,
    output_path     = output_file_path_PL_RASTER,
    raster_name     = "density_all"
)

print(meta_v2)


In [ ]:
# ─── Normalisation et plot ────────────────────────────────────────────────────
d_max_v2          = compute_scale([raster_mean_clipped_v2], clip_percentile, mode=norm_mode)
raster_mean_norm_v2 = normalize_raster(raster_mean_clipped_v2, d_max_v2, mode=norm_mode)

plot_density_map(
    raster_mean_norm_v2,
    title=f"Mean Daily Pedestrian Density — All users\nn_users={meta_v2['n_users_valid']} | norm={norm_mode} | P{clip_percentile}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec
)

plot_density_map(
    raster_mean_norm_v2,
    title=f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\nAll users | norm={norm_mode} | P{clip_percentile}",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    focus=focus_commune,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    margin=margin
)

In [ ]:
# ─── Distribution des valeurs de pixels — raster all users ───────────────────
vals_all    = raster_mean_clipped_v2[~np.isnan(raster_mean_clipped_v2)]
vals_active = vals_all[vals_all > 0]

p95  = np.percentile(vals_active, 95)
p99  = np.percentile(vals_active, 99)
p100 = vals_active.max()

print(f"pixels canton total : {len(vals_all):,}")
print(f"pixels actifs (>0)  : {len(vals_active):,}")
print(f"pixels zéro         : {(vals_all == 0).sum():,}")
print(f"min                 : {vals_active.min():.6f}")
print(f"max (P100)          : {p100:.6f}")
print(f"mean                : {vals_active.mean():.6f}")
print(f"median              : {np.median(vals_active):.6f}")
print(f"P50                 : {np.percentile(vals_active, 50):.6f}")
print(f"P75                 : {np.percentile(vals_active, 75):.6f}")
print(f"P90                 : {np.percentile(vals_active, 90):.6f}")
print(f"P95                 : {p95:.6f}")
print(f"P99                 : {p99:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Pixel value distribution — All users raster", fontsize=13, fontweight="bold")

# ─── Panneau gauche : distribution complète ───────────────────────────────────
axes[0].hist(vals_active, bins=100, color="steelblue", edgecolor="none")
axes[0].axvline(p95,  color="red",    linestyle="--", label=f"P95  = {p95:.4f}")
axes[0].axvline(p99,  color="orange", linestyle="--", label=f"P99  = {p99:.4f}")
axes[0].axvline(p100, color="black",  linestyle="--", label=f"P100 = {p100:.4f}")
axes[0].set_title("Full distribution (active pixels)")
axes[0].set_xlabel("Pixel value (legs · day⁻¹ · user⁻¹)")
axes[0].set_ylabel("Number of pixels")
axes[0].legend(fontsize=8)

# ─── Panneau droit : queue de distribution (> P95) ───────────────────────────
vals_tail = vals_active[vals_active >= p95]
axes[1].hist(vals_tail, bins=100, color="salmon", edgecolor="none")
axes[1].axvline(p95,  color="red",    linestyle="--", label=f"P95  = {p95:.4f}")
axes[1].axvline(p99,  color="orange", linestyle="--", label=f"P99  = {p99:.4f}")
axes[1].axvline(p100, color="black",  linestyle="--", label=f"P100 = {p100:.4f}")
axes[1].set_title("Tail distribution (pixels ≥ P95)")
axes[1].set_xlabel("Pixel value (legs · day⁻¹ · user⁻¹)")
axes[1].set_ylabel("Number of pixels")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
filepath = output_file_path_PL_RASTER + "density_all_day.tif"

with rasterio.open(filepath) as src:
    raster = src.read(1).astype("float32")

pixels_actifs = raster[~np.isnan(raster) & (raster > 0)]

p50 = np.percentile(pixels_actifs, 50)
p75 = np.percentile(pixels_actifs, 75)
p90 = np.percentile(pixels_actifs, 90)
p95 = np.percentile(pixels_actifs, 95)
p99 = np.percentile(pixels_actifs, 99)

print(f"pixels actifs : {len(pixels_actifs):,}")
print(f"min  : {pixels_actifs.min():.6f}")
print(f"max  : {pixels_actifs.max():.6f}")
print(f"mean : {pixels_actifs.mean():.6f}")
print(f"P50  : {p50:.6f}  → {(pixels_actifs > p50).sum():,} pixels au-dessus ({100*(pixels_actifs > p50).mean():.1f}%)")
print(f"P75  : {p75:.6f}  → {(pixels_actifs > p75).sum():,} pixels au-dessus ({100*(pixels_actifs > p75).mean():.1f}%)")
print(f"P90  : {p90:.6f}  → {(pixels_actifs > p90).sum():,} pixels au-dessus ({100*(pixels_actifs > p90).mean():.1f}%)")
print(f"P95  : {p95:.6f}  → {(pixels_actifs > p95).sum():,} pixels au-dessus ({100*(pixels_actifs > p95).mean():.1f}%)")
print(f"P99  : {p99:.6f}  → {(pixels_actifs > p99).sum():,} pixels au-dessus ({100*(pixels_actifs > p99).mean():.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribution des pixels actifs — density_all_day", fontsize=13, fontweight="bold")

for ax, p_val, p_name in zip(axes, [p95, p99], ["P95", "P99"]):
    ax.hist(pixels_actifs, bins=200, color="steelblue", alpha=0.7)
    ax.axvline(p_val, color="red", linewidth=1.5, label=f"{p_name} = {p_val:.6f}")
    ax.set_xlim(0, p_val * 1.5)  # zoom autour du percentile
    ax.set_xlabel("Densité (passages/jour)")
    ax.set_ylabel("Nombre de pixels")
    ax.set_title(f"Zoom autour du {p_name}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Distribution après clipping + renormalisation — density_all", fontsize=13, fontweight="bold")

for ax, p_val, p_name in zip(axes, [p50, p95, p99], ["P50", "P95", "P99"]):
    
    pixels_clipped = np.clip(pixels_actifs, 0, p_val) / p_val
    n_clipped = (pixels_clipped == 1.0).sum()

    print(f"{p_name} : {n_clipped:,} pixels clippés à 1 ({100*(pixels_clipped == 1.0).mean():.1f}%)")

    ax.hist(pixels_clipped, bins=200, color="steelblue", alpha=0.7)
    ax.set_yscale("log")
    ax.axvline(1.0, color="red", linewidth=1.5, linestyle="--", label=f"clip à {p_name}")
    ax.axhline(n_clipped, color="orange", linewidth=1.5, linestyle="--", label=f"n clippés = {n_clipped:,}")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Densité normalisée [0, 1]")
    ax.set_ylabel("Nombre de pixels (log)")
    ax.set_title(f"Clipping à {p_name} = {p_val:.6f}")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(raster, origin="upper", cmap="viridis")
plt.colorbar(im, ax=ax, label="Densité (passages/jour/user)")
ax.set_title("density_all_day")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Boucle par groupe ────────────────────────────────────────────────────────
rasters_gender = {}
metas_gender   = {}

for gender_name, gdr_value in gender_filters:
    print(f"\n{'='*50}")
    print(f"Processing : {gender_name}...")

    gdf_gender = gdf if gdr_value is None else gdf[gdf["gdr"] == gdr_value].copy()

    print(f"  → {len(gdf_gender)} legs / {gdf_gender['user_id_fors'].nunique()} users")

    rasters_gender[gender_name], metas_gender[gender_name] = build_density_raster_v2(
        gdf_group       = gdf_gender,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = True,
        output_path     = output_file_path_PL_RASTER,
        raster_name     =f"density_{gender_name}",
    )

    del gdf_gender

# ─── Normalisation commune ────────────────────────────────────────────────────
gender_max = compute_scale(
    [rasters_gender[g] for g, _ in gender_filters],
    clip_percentile, mode=norm_mode
)

print(f"\ngender_max : {gender_max:.6f}")

# ─── Plot par groupe ──────────────────────────────────────────────────────────
for gender_name, _ in gender_filters:
    label        = gender_labels[gender_name]
    density_norm = normalize_raster(rasters_gender[gender_name], gender_max, mode=norm_mode)

    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — {label}\nn_users={metas_gender[gender_name]['n_users_valid']}",
        extent=extent,
        canton_GE=canton_GE,
        girec=girec
    )

    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\n{label} | n_users={metas_gender[gender_name]['n_users_valid']}",
        extent=extent,
        canton_GE=canton_GE,
        girec=girec,
        focus=focus_commune,
        zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
        margin=margin
    )

    del density_norm

plot_user_contribution(
    gdf_group     = gdf,
    group_filters = gender_filters,
    group_labels  = gender_labels,
    group_col     = "gdr",
    title         = "User contribution — Gender"
)

del rasters_gender


In [ ]:
# ─── Boucle par groupe âge ────────────────────────────────────────────────────
rasters_age = {}
metas_age   = {}

for age_name, age_value in age_filters:
    print(f"\n{'='*50}")
    print(f"Processing : {age_name}...")

    gdf_age = gdf[gdf["age_fr_grouped"] == age_value].copy()
    print(f"  → {len(gdf_age)} legs / {gdf_age['user_id_fors'].nunique()} users")

    rasters_age[age_name], metas_age[age_name] = build_density_raster_v2(
        gdf_group       = gdf_age,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = True,
        output_path     = output_file_path_PL_RASTER,
        raster_name     = f"density_{age_name}",
    )

    del gdf_age

# ─── Normalisation commune âge ────────────────────────────────────────────────
age_max = compute_scale(
    [rasters_age[a] for a, _ in age_filters],
    clip_percentile, mode=norm_mode
)
print(f"\nage_max : {age_max:.6f}")

# ─── Plot âge ─────────────────────────────────────────────────────────────────
for age_name, _ in age_filters:
    label        = age_labels[age_name]
    density_norm = normalize_raster(rasters_age[age_name], age_max, mode=norm_mode)

    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — {label}\nn_users={metas_age[age_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec
    )
    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\n{label} | n_users={metas_age[age_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec,
        focus=focus_commune,
        zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
        margin=margin
    )

    del density_norm

# ─── Contribution âge ─────────────────────────────────────────────────────────
plot_user_contribution(
    gdf_group     = gdf,
    group_filters = age_filters,
    group_labels  = age_labels,
    group_col     = "age_fr_grouped",
    title         = "User contribution — Age"
)

del rasters_age

In [ ]:
# ─── Boucle par groupe revenu ─────────────────────────────────────────────────
rasters_income = {}
metas_income   = {}

for income_name, _ in income_filters:  # ← on ignore income_value
    en_label   = labels_all_groups[income_name]  # ← "Low income", "Median income", "High income"
    gdf_income = gdf[gdf["income_class"] == en_label].copy()
    print(f"  → {len(gdf_income)} legs / {gdf_income['user_id_fors'].nunique()} users")

    rasters_income[income_name], metas_income[income_name] = build_density_raster_v2(
        gdf_group       = gdf_income,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = True,
        output_path     = output_file_path_PL_RASTER,
        raster_name     = f"density_{income_name}",
    )

    del gdf_income

# ─── Normalisation commune revenu ─────────────────────────────────────────────
income_max = compute_scale(
    [rasters_income[inc] for inc, _ in income_filters],
    clip_percentile, mode=norm_mode
)
print(f"\nincome_max : {income_max:.6f}")

# ─── Plot revenu ──────────────────────────────────────────────────────────────
for income_name, _ in income_filters:
    label        = income_labels[income_name]
    density_norm = normalize_raster(rasters_income[income_name], income_max, mode=norm_mode)

    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — {label}\nn_users={metas_income[income_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec
    )
    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\n{label} | n_users={metas_income[income_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec,
        focus=focus_commune,
        zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
        margin=margin
    )

    del density_norm

# ─── Contribution revenu ──────────────────────────────────────────────────────
plot_user_contribution(
    gdf_group     = gdf,
    group_filters = income_filters_en,
    group_labels  = income_labels,
    group_col     = "income_class",
    title         = "User contribution — Income"
)

del rasters_income

In [ ]:
# ─── Boucle par groupe voiture ────────────────────────────────────────────────
rasters_car = {}
metas_car   = {}

for car_name, car_value in car_filters_en:
    print(f"\n{'='*50}")
    print(f"Processing : {car_name}...")

    gdf_car = gdf[gdf["has_car"] == car_value].copy()
    print(f"  → {len(gdf_car)} legs / {gdf_car['user_id_fors'].nunique()} users")

    rasters_car[car_name], metas_car[car_name] = build_density_raster_v2(
        gdf_group       = gdf_car,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = True,
        output_path     = output_file_path_PL_RASTER,
        raster_name     = f"density_{car_name}",
    )

    del gdf_car

# ─── Normalisation commune voiture ────────────────────────────────────────────
car_max = compute_scale(
    [rasters_car[c] for c, _ in car_filters_en],
    clip_percentile, mode=norm_mode
)
print(f"\ncar_max : {car_max:.6f}")

# ─── Plot voiture ─────────────────────────────────────────────────────────────
for car_name, _ in car_filters:
    label        = labels_all_groups[car_name]
    density_norm = normalize_raster(rasters_car[car_name], car_max, mode=norm_mode)

    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — {label}\nn_users={metas_car[car_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec
    )
    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\n{label} | n_users={metas_car[car_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec,
        focus=focus_commune,
        zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
        margin=margin
    )

    del density_norm

# ─── Contribution voiture ─────────────────────────────────────────────────────
plot_user_contribution(
    gdf_group     = gdf,
    group_filters = car_filters_en,
    group_labels  = {k: labels_all_groups[k] for k, _ in car_filters},
    group_col     = "has_car",
    title         = "User contribution — Car ownership"
)

del rasters_car

In [ ]:
rasters_tp = {}
metas_tp   = {}

for tp_name, tp_value in tp_filters:
    print(f"\n{'='*50}")
    print(f"Processing : {tp_name}...")

    gdf_tp = gdf[gdf["tp_level"] == tp_value].copy()
    print(f"  → {len(gdf_tp)} legs / {gdf_tp['user_id_fors'].nunique()} users")

    rasters_tp[tp_name], metas_tp[tp_name] = build_density_raster_v2(
        gdf_group       = gdf_tp,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = True,
        output_path     = output_file_path_PL_RASTER,
        raster_name     = f"density_{tp_name}",
    )

    del gdf_tp

tp_max = compute_scale(
    [rasters_tp[t] for t, _ in tp_filters],
    clip_percentile, mode=norm_mode
)

for tp_name, _ in tp_filters:
    label        = tp_labels[tp_name]
    density_norm = normalize_raster(rasters_tp[tp_name], tp_max, mode=norm_mode)

    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — {label}\nn_users={metas_tp[tp_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec
    )
    plot_density_map(
        density_norm,
        title=f"Mean Daily Pedestrian Density — Zoom {focus_commune_name}\n{label} | n_users={metas_tp[tp_name]['n_users_valid']}",
        extent=extent, canton_GE=canton_GE, girec=girec,
        focus=focus_commune,
        zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
        margin=margin
    )

    del density_norm

plot_user_contribution(
    gdf_group     = gdf,
    group_filters = tp_filters,
    group_labels  = tp_labels,
    group_col     = "tp_level",
    title         = "User contribution — PT subscription"
)

del rasters_tp

# TEMPORAL RASTER

In [ ]:
# ─── Calcul des rasters par créneau horaire ───────────────────────────────────
rasters_hourly = {}
metas_hourly   = {}

for slot_name, h_start, m_start, h_end, m_end in time_slots:
    print(f"\n{'─'*40}")
    print(f"Processing : {slot_name} ({h_start:02d}:{m_start:02d} - {h_end:02d}:{m_end:02d})...")

    if slot_name == "all_day":
        gdf_slot = gdf.copy()
    else:
        start_minutes = h_start * 60 + m_start
        end_minutes   = h_end   * 60 + m_end
        gdf_slot = gdf[
            (gdf["started_at_local"].dt.hour * 60 + gdf["started_at_local"].dt.minute >= start_minutes) &
            (gdf["started_at_local"].dt.hour * 60 + gdf["started_at_local"].dt.minute <= end_minutes)
        ].copy()

    print(f"  → {len(gdf_slot)} legs / {gdf_slot['user_id_fors'].nunique()} users")

    rasters_hourly[slot_name], metas_hourly[slot_name] = build_density_raster_v2(
        gdf_group       = gdf_slot,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = False,
        output_path     = output_file_path_PL_RASTER,
        raster_name     = f"density_{slot_name}", 
    )

    del gdf_slot 

print(f"\n✓ {len(rasters_hourly)} créneaux calculés")

# ─── Normalisation commune sur tous les créneaux ──────────────────────────────
slot_names_only = [s for s, _, _, _, _ in time_slots if s != "all_day"]

hourly_max = compute_scale(
    [rasters_hourly[s] for s in slot_names_only if (rasters_hourly[s] > 0).any()],
    clip_percentile, mode=norm_mode
)
all_day_max = compute_scale(
    [rasters_hourly["all_day"]],
    clip_percentile, mode=norm_mode
)

print(f"hourly_max  : {hourly_max:.6f}")
print(f"all_day_max : {all_day_max:.6f}")

# ─── Small multiples — tous les créneaux ─────────────────────────────────────
plot_small_multiples(
    rasters_dict = rasters_hourly,
    slot_names   = slot_names_only,
    labels_dict  = slot_labels,
    title        = "Mean Daily Pedestrian Density — Time slots | All users",
    extent       = extent,
    canton_GE    = canton_GE,
    girec        = girec,
    d_max        = hourly_max,
    ncols        = 4,
    plot_individual = True
)

# ─── Small multiples — zoom commune ───────────────────────────────────────────
plot_small_multiples(
    rasters_dict = rasters_hourly,
    slot_names   = slot_names_only,
    labels_dict  = slot_labels,
    title        = f"Mean Daily Pedestrian Density — Time slots | Zoom {focus_commune_name}",
    extent       = extent,
    canton_GE    = canton_GE,
    girec        = girec,
    d_max        = hourly_max,
    zoom_bounds  = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin       = margin,
    ncols        = 4,
    plot_individual = True
)

# ─── Plot all_day séparé ──────────────────────────────────────────────────────
density_norm_all_day = normalize_raster(rasters_hourly["all_day"], all_day_max, mode=norm_mode)

plot_density_map(
    density_norm_all_day,
    title=f"Mean Daily Pedestrian Density — All day\nn_users={metas_hourly['all_day']['n_users_valid']}",
    extent=extent, canton_GE=canton_GE, girec=girec
)

plot_density_map(
    density_norm_all_day,
    title=f"Mean Daily Pedestrian Density — All day | Zoom {focus_commune_name}",
    extent=extent, canton_GE=canton_GE, girec=girec,
    focus=focus_commune,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    margin=margin
)

del density_norm_all_day

In [ ]:
# ─── Définition des créneaux horaires individuels ─────────────────────────────
hourly_slots = [("all_day", 0, 0, 23, 59)] + [
    (f"{h:02d}h", h, 0, h+1, 0) for h in range(6, 22)
]

hourly_labels = {f"{h:02d}h": f"{h:02d}h - {h+1:02d}h" for h in range(6, 22)}
hourly_labels["all_day"] = "All day"

hourly_names = [s for s, *_ in hourly_slots if s != "all_day"]

# ─── Calcul + export des rasters horaires ────────────────────────────────────
rasters_hourly_h = {}
metas_hourly_h   = {}

for slot_name, h_start, m_start, h_end, m_end in hourly_slots:
    print(f"Processing : {slot_name}...", end=" ")

    if slot_name == "all_day":
        gdf_slot = gdf.copy()
    else:
        gdf_slot = gdf[gdf["started_at_local"].dt.hour == h_start].copy()

    print(f"→ {len(gdf_slot)} legs / {gdf_slot['user_id_fors'].nunique()} users")

    rasters_hourly_h[slot_name], metas_hourly_h[slot_name] = build_density_raster_v2(
        gdf_group       = gdf_slot,
        raster_height   = raster_height,
        raster_width    = raster_width,
        transform       = transform,
        canton_mask     = canton_GE_mask,
        n_days_per_user = n_days_per_user,
        sigma           = 1,
        verbose         = False,
        output_path     = output_file_path_PL_RASTER,
        raster_name     = f"density_hourly_{slot_name}"
    )

    del gdf_slot

print(f"\n✓ {len(rasters_hourly_h)} rasters horaires calculés")

# ─── Normalisation commune ────────────────────────────────────────────────────
hourly_h_max = compute_scale(
    [rasters_hourly_h[s] for s in hourly_names if (rasters_hourly_h[s] > 0).any()],
    clip_percentile, mode=norm_mode
)
print(f"hourly_h_max : {hourly_h_max:.6f}")

In [ ]:
# ─── Small multiples ──────────────────────────────────────────────────────────
plot_small_multiples(
    rasters_dict    = rasters_hourly_h,
    slot_names      = hourly_names,
    labels_dict     = hourly_labels,
    title           = "Mean Daily Pedestrian Density — Hourly | All users",
    extent          = extent,
    canton_GE       = canton_GE,
    girec           = girec,
    d_max           = hourly_h_max,
    ncols           = 4,
    plot_individual = False  # ← pas de plots individuels
)

plot_small_multiples(
    rasters_dict    = rasters_hourly_h,
    slot_names      = hourly_names,
    labels_dict     = hourly_labels,
    title           = f"Mean Daily Pedestrian Density — Hourly | Zoom {focus_commune_name}",
    extent          = extent,
    canton_GE       = canton_GE,
    girec           = girec,
    d_max           = hourly_h_max,
    zoom_bounds     = (xmin_z, ymin_z, xmax_z, ymax_z),
    margin          = margin,
    ncols           = 4,
    plot_individual = False
)

In [ ]:
# ─── Libération RAM ───────────────────────────────────────────────────────────
del rasters_hourly_h
print("RAM libérée")

In [ ]:
filepath_06h = output_file_path_PL_RASTER + "density_morning_early.tif"

with rasterio.open(filepath_06h) as src:
    raster_06h = src.read(1).astype("float32")

pixels_actifs_06h = raster_06h[~np.isnan(raster_06h) & (raster_06h > 0)]

print(f"pixels actifs : {len(pixels_actifs_06h):,}")
print(f"min  : {pixels_actifs_06h.min():.6f}")
print(f"max  : {pixels_actifs_06h.max():.6f}")

d_max = compute_scale([raster_06h], clip_percentile=100, mode=norm_mode)
raster_06h_norm = normalize_raster(raster_06h, d_max, mode=norm_mode)

plot_density_map(
    raster_06h_norm,
    title="density_morning_early (05:00 - 07:00)",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    clip_percentile_label=100
)

In [ ]:
filepath_06h = output_file_path_PL_RASTER + "density_morning_early.tif"

with rasterio.open(filepath_06h) as src:
    raster_06h = src.read(1).astype("float32")

pixels_actifs_06h = raster_06h[~np.isnan(raster_06h) & (raster_06h > 0)]

print(f"pixels actifs : {len(pixels_actifs_06h):,}")
print(f"min  : {pixels_actifs_06h.min():.6f}")
print(f"max  : {pixels_actifs_06h.max():.6f}")

d_max = compute_scale([raster_06h], clip_percentile=99, mode=norm_mode)
raster_06h_norm = normalize_raster(raster_06h, d_max, mode=norm_mode)

print(clip_percentile)

plot_density_map(
    raster_06h_norm,
    title="density_morning_early (05:00 - 07:00)",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    clip_percentile_label=99
)

In [ ]:
filepath_06h = output_file_path_PL_RASTER + "density_morning_early.tif"

with rasterio.open(filepath_06h) as src:
    raster_06h = src.read(1).astype("float32")

pixels_actifs_06h = raster_06h[~np.isnan(raster_06h) & (raster_06h > 0)]

print(f"pixels actifs : {len(pixels_actifs_06h):,}")
print(f"min  : {pixels_actifs_06h.min():.6f}")
print(f"max  : {pixels_actifs_06h.max():.6f}")

d_max = compute_scale([raster_06h], clip_percentile=95, mode=norm_mode)
raster_06h_norm = normalize_raster(raster_06h, d_max, mode=norm_mode)

plot_density_map(
    raster_06h_norm,
    title="density_morning_early (05:00 - 07:00)",
    extent=extent,
    canton_GE=canton_GE,
    girec=girec,
    zoom_bounds=(xmin_z, ymin_z, xmax_z, ymax_z),
    clip_percentile_label=95
)